Feature engeneering

In [1]:
import pandas as pd
import numpy as np

# Load database saved on notebook 01 before
df = pd.read_csv('data/processed/orders_with_target.csv', parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

# Load auxliary tables
order_items = pd.read_csv('data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('data/raw/olist_order_payments_dataset.csv')
customers = pd.read_csv('data/raw/olist_customers_dataset.csv')
products = pd.read_csv('data/raw/olist_products_dataset.csv')

print(f"Load database: {df.shape}")
print(f"Available columns: {list(df.columns)}")

Load database: (97007, 11)
Available columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'review_score', 'churn_risk']


In [5]:
# Time features
df['days_to_approve'] = (
    df['order_approved_at'] - df['order_purchase_timestamp']
).dt.total_seconds() / 3600  # convert to hours

df['days_to_carrier'] = (
    df['order_delivered_carrier_date'] - df['order_approved_at']
).dt.days

df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek  # 0=monday, 6=sunday
df['purchase_month'] = df['order_purchase_timestamp'].dt.month

print("Time features created!")
print(df[['days_to_approve', 'days_to_carrier', 'purchase_hour', 
          'purchase_dayofweek', 'purchase_month']].describe())

Time features created!
       days_to_approve  days_to_carrier  purchase_hour  purchase_dayofweek  \
count     96993.000000     96991.000000   97007.000000        97007.000000   
mean         10.280161         2.297553      14.772934            2.757141   
std          20.526056         3.545382       5.329760            1.967007   
min           0.000000      -172.000000       0.000000            0.000000   
25%           0.215278         0.000000      11.000000            1.000000   
50%           0.343333         1.000000      15.000000            3.000000   
75%          14.541944         3.000000      19.000000            4.000000   
max         741.443611       125.000000      23.000000            6.000000   

       purchase_month  
count    97007.000000  
mean         6.030152  
std          3.229851  
min          1.000000  
25%          3.000000  
50%          6.000000  
75%          8.000000  
max         12.000000  


In [6]:
# Financial features

# Aggregating order_items by order_id
items_agg = order_items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    n_items=('order_item_id', 'count')
).reset_index()

# Calculating freight_ratio (what portion of the total value is freight)
items_agg['freight_ratio'] = items_agg['total_freight'] / (
    items_agg['total_price'] + items_agg['total_freight']
)

# Aggregating payments by order_id
payments_agg = payments.groupby('order_id').agg(
    payment_installments=('payment_installments', 'max'),
    total_payment=('payment_value', 'sum'),
    n_payment_types=('payment_type', 'nunique')
).reset_index()

# Merge no df principal
df = df.merge(items_agg, on='order_id', how='left')
df = df.merge(payments_agg, on='order_id', how='left')

print("Financial features created!")
print(df[['total_price', 'total_freight', 'freight_ratio', 
          'n_items', 'payment_installments', 'n_payment_types']].describe())

Financial features created!
        total_price  total_freight  freight_ratio       n_items  \
count  97007.000000   97007.000000   97007.000000  97007.000000   
mean     136.895653      22.780098       0.209058      1.142598   
std      208.698701      21.532577       0.125569      0.540014   
min        0.850000       0.000000       0.000000      1.000000   
25%       45.900000      13.850000       0.116687      1.000000   
50%       86.000000      17.170000       0.183264      1.000000   
75%      149.900000      24.020000       0.275803      1.000000   
max    13440.000000    1794.960000       0.955451     21.000000   

       payment_installments  n_payment_types  
count          97006.000000     97006.000000  
mean               2.931551         1.022617  
std                2.715884         0.148680  
min                0.000000         1.000000  
25%                1.000000         1.000000  
50%                2.000000         1.000000  
75%                4.000000         1.0

In [7]:
# Location features 
df = df.merge(
    customers[['customer_id', 'customer_state']],
    on='customer_id',
    how='left'
)

# Retrieving seller status (via order_items + sellers)
sellers = pd.read_csv('data/raw/olist_sellers_dataset.csv')

items_seller = order_items[['order_id', 'seller_id']].drop_duplicates('order_id')
items_seller = items_seller.merge(
    sellers[['seller_id', 'seller_state']],
    on='seller_id',
    how='left'
)

df = df.merge(items_seller[['order_id', 'seller_state']], on='order_id', how='left')

# Feature: interestadual seller
df['interstate'] = (df['customer_state'] != df['seller_state']).astype(int)

print("Location features created!")
print(f"\nInterstate sales: {df['interstate'].sum()} ({df['interstate'].mean()*100:.1f}%)")
print(f"\nTop 5 customer states:\n{df['customer_state'].value_counts().head()}")

Location features created!

Interstate sales: 62129 (64.0%)

Top 5 customer states:
customer_state
SP    40712
RJ    12420
MG    11423
RS     5382
PR     4942
Name: count, dtype: int64


In [8]:
# Product category features
category_translation = pd.read_csv('data/raw/product_category_name_translation.csv')

# Merging products with category translation
products = products.merge(category_translation, on='product_category_name', how='left')

# Getting product category per order (first item only)
items_category = order_items[['order_id', 'product_id']].drop_duplicates('order_id')
items_category = items_category.merge(
    products[['product_id', 'product_category_name_english']],
    on='product_id',
    how='left'
)

df = df.merge(items_category[['order_id', 'product_category_name_english']], 
              on='order_id', how='left')

# Grouping categories with less than 500 orders into 'other'
category_counts = df['product_category_name_english'].value_counts()
rare_categories = category_counts[category_counts < 500].index
df['product_category'] = df['product_category_name_english'].apply(
    lambda x: 'other' if x in rare_categories else x
)

print("Product category features created!")
print(f"\nTop 10 categories:\n{df['product_category'].value_counts().head(10)}")
print(f"\nTotal categories: {df['product_category'].nunique()}")

Product category features created!

Top 10 categories:
product_category
bed_bath_table           9282
health_beauty            8661
sports_leisure           7541
other                    7356
computers_accessories    6551
furniture_decor          6268
housewares               5709
watches_gifts            5482
telephony                4080
auto                     3808
Name: count, dtype: int64

Total categories: 26


In [9]:
# Selecting final features for modeling
feature_cols = [
    # Time features
    'days_to_approve',
    'days_to_carrier',
    'purchase_hour',
    'purchase_dayofweek',
    'purchase_month',
    # Financial features
    'total_price',
    'total_freight',
    'freight_ratio',
    'n_items',
    'payment_installments',
    'n_payment_types',
    # Location features
    'interstate',
    # Category feature
    'product_category',
    # Target
    'churn_risk'
]

df_model = df[feature_cols].copy()

print(f"Final dataset shape: {df_model.shape}")
print(f"\nMissing values:\n{df_model.isnull().sum()[df_model.isnull().sum() > 0]}")

Final dataset shape: (97007, 14)

Missing values:
days_to_approve           14
days_to_carrier           16
payment_installments       1
n_payment_types            1
product_category        1386
dtype: int64


In [10]:
# Handling missing values

# Numerical features: filling with median (robust to outliers)
num_cols = ['days_to_approve', 'days_to_carrier', 'payment_installments', 'n_payment_types']
for col in num_cols:
    median_val = df_model[col].median()
    df_model[col] = df_model[col].fillna(median_val)

# Product category: filling with 'unknown'
df_model['product_category'] = df_model['product_category'].fillna('unknown')

print("Missing values handled!")
print(f"\nRemaining missing values: {df_model.isnull().sum().sum()}")
print(f"\nDataset shape: {df_model.shape}")

Missing values handled!

Remaining missing values: 0

Dataset shape: (97007, 14)


In [ ]:
import os

# Saving final dataset for modeling
os.makedirs('data/processed', exist_ok=True)
df_model.to_csv('data/processed/features.csv', index=False)

print(" Dataset saved at data/processed/features.csv")
print(f"\nFinal features:")
for col in df_model.columns:
    print(f"  - {col}: {df_model[col].dtype}")

✅ Dataset saved at data/processed/features.csv

Final features:
  - days_to_approve: float64
  - days_to_carrier: float64
  - purchase_hour: int32
  - purchase_dayofweek: int32
  - purchase_month: int32
  - total_price: float64
  - total_freight: float64
  - freight_ratio: float64
  - n_items: int64
  - payment_installments: float64
  - n_payment_types: float64
  - interstate: int32
  - product_category: object
  - churn_risk: int64
